# Slow-Light Black Hole Imaging from GRMHD Simulations

In this notebook, we move beyond the fast-light approximation and explore slow-light radiative transfer. 
This approach accounts for the time-dependent evolution of the plasma, enabling more physically consistent 
black hole images derived from GRMHD simulations.

- In Jipole, this is done by enabling the variable SLOW_LIGHT = true

In [1]:
const MODEL = "iharm"
const MBH = 6.2e9
const SLOW_LIGHT = true
include("../src/main.jl");

Using model: iharm, change src/set_globals.jl to modify.
Adding slowlight.jl file...


We must account for the time evolution of the plasma during photon propagation. In practice, this means that a single simulation snapshot is no longer sufficient.

Instead, slow-light radiative transfer requires access to multiple GRMHD dump files, each corresponding to a different simulation time. As photons travel through the domain, their trajectories intersect the simulation at different times, and the local fluid quantities (e.g., density, temperature, magnetic field) must be interpolated from the appropriate dumps.

In this notebook, we therefore work with a sequence of simulation outputs stored as:


In [2]:
const all_dumps_path = "/home/pedro/kharma/iharm3d_out/tmp.%05d.h5"

"/home/pedro/kharma/iharm3d_out/tmp.%05d.h5"

To perform slow-light, we must specify both the temporal range of the simulation data and how frequently images are produced. 

The parameter `dump_max` sets the extent of the available simulation snapshots, meaning that the radiative transfer will interpolate fluid quantities using dumps from the initial index up to this maximum value. 

In this example, the evolution of the plasma is sampled from dump 0 through dump 10, providing a finite time window over which photon trajectories are evaluated.

The parameter `ImageCadence` controls how often images are generated in physical time. For instance, a value of 10 corresponds to producing one image every \(10\ r_g / c\), where \(r_g\) is the gravitational radius. This defines the temporal resolution of the resulting sequence of images and ultimately determines how smoothly the time variability of the system is captured.


All of this information is stored in an `OfSlowLight` structure, which manages how simulation data is loaded and interpolated in time during the ray-tracing process.

At any given moment, only a small subset of the available dumps is kept in memory to remain efficient. In this implementation, we always keep three dump files loaded. These define a sliding time window over which interpolation is performed. The variables `tA` and `tB` correspond to the times of two consecutive dumps (for example, dump 0 and dump 1). If a photon intersects the simulation at a time between `tA` and `tB`, the fluid quantities are obtained by interpolating between these two snapshots. As the photon propagates forward in time and moves beyond `tB`, the window shifts, and interpolation is then performed between the next pair of dumps (e.g., dumps 1 and 2, then 2 and 3, and so on).

The field `current_dumps_path` keeps track of the next dump file to be loaded into memory as this sliding window advances. This allows the code to progressively stream simulation data without loading all dumps at once.

`tf` represents the final time of the simulation data, corresponding to the last available dump as defined by `dump_max`. This sets the upper temporal boundary for the slow-light calculation.

In [3]:
const dump_max = 10
const ImageCadence = 10 
params_slowlight = OfSlowLight(dump_max, 0, ImageCadence, 0.0, 0.0, 0.0, "")
params_slowlight.current_dumps_path = update_dump_path()

"/home/pedro/kharma/iharm3d_out/tmp.00000.h5"

In [4]:
trat_large = 20. 
const trat_small = 1. 
const beta_crit = 1.0 
const th_beg = 1.74e-2 
const sigma_cut = 1.0 
const sigma_cut_high = -1.0;

In [5]:
const params = read_header(params_slowlight.current_dumps_path);

Initializing grid from: /home/pedro/kharma/iharm3d_out/tmp.00000.h5


custom electron model loaded from dump file...
Using mixed tp_over_te with trat_small = 1, trat_large = 20, and beta_crit = 1
Using Funky Modified Kerr-Schild coordinates FMKS
MKS parameters a: 0.937500 hslope: 0.300000 Rin: 1.001876 Rout: 1000.000000
FMKS parameters poly_xt: 0.820000 poly_alpha: 14.000000 mks_smooth: 0.500000 poly_norm: 0.757817
Grid start (startx): 1.874000951149755e-03, 0.000000000000000e+00, 0.000000000000000e+00 stop (stopx): 6.907755278982137e+00, 1.000000000000000e+00, 6.283185307179586e+00
grid dx: 5.395219748461709e-02, 7.812500000000000e-03, 6.283185307179586e+00


We begin by initializing a vector of simulation data containing three dump files and loading the first three snapshots into memory. These initial dumps define the starting interpolation window for the slow-light calculation.

As photon trajectories advance in time, this window must be updated to ensure that the appropriate simulation data is always available. All loading of new dump files and shifting of the sliding window is handled internally by the `update_data!` function (defined in `slowlight.jl`). This routine takes care of discarding old snapshots, loading new ones, and maintaining the correct ordering for time interpolation throughout the radiative transfer.

In [6]:
const simulation_data = Vector{IharmData}(undef, 3)

# everytime you load a file slow_light mode will automatically advance to the next one
simulation_data[1] = load_data(params_slowlight.current_dumps_path, trat_large)
simulation_data[2] = load_data(params_slowlight.current_dumps_path, trat_large)
simulation_data[3] = load_data(params_slowlight.current_dumps_path, trat_large)

params_slowlight.tA = simulation_data[1].t;
params_slowlight.tB = simulation_data[2].t;

params_slowlight.tf = get_specific_dump_time(params_slowlight.dump_max);

Loading data from '/home/pedro/kharma/iharm3d_out/tmp.00000.h5' into 'iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)
Loading data from '/home/pedro/kharma/iharm3d_out/tmp.00001.h5' into 'iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)
Loading data from '/home/pedro/kharma/iharm3d_out/tmp.00002.h5' into 'iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)


In [7]:
# Observer distance in gravitational radii (Rg)
const ro = 1000.0

# Inclination angle (deg) — angle between the observer and the BH spin axis
const th = 163.0

# Azimuthal angle (deg) — rotation around the system
const phi = 0.0

# Image resolution — total geodesics traced = res^2
const res = 128
const pixels_x = 128
const pixels_y = 128

# Distance to the source (in parsecs, converted to code units)
const SourceD = 16.9e6 * PC

# Radius where ray integration stops
const Rstop = 100.0

# Event horizon radius for a Kerr black hole
const Rh = 1 + sqrt(1. - params.a * params.a)

# Observing frequency (Hz), e.g. 230 GHz for EHT-like images
const freq = 230e9

# Image plane size (in Rg), scaled from physical distance
const DXsize = SourceD / L_unit / MUAS_PER_RAD * 160
const DYsize = SourceD / L_unit / MUAS_PER_RAD * 160

# Field of view (radians)
const fovx = DXsize / ro
const fovy = DYsize / ro

# Image offsets (can be used to shift the camera)
const xoff = 0.0
const yoff = 0.0


0.0

## Geodesic Ray-Tracing and Time Boundary Estimation

In this stage, we perform the core ray-tracing calculation to construct the final image from the GRMHD simulation. The camera position is first defined in the native coordinate system of the spacetime, and the observed frequency is converted into a dimensionless unit system appropriate for the geodesic integration.

We allocate memory for the auxiliary arrays to track the number of integration steps per pixel and the number of midplane crossings. Because the computation is parallelized across multiple threads, each thread is assigned its own scratch space to store geodesic trajectories. Each trajectory is represented as a sequence of spacetime points and wavevectors, with a fixed maximum number of integration steps to ensure memory safety.

Following the conventions used in ipole, we compute two diagnostic time boundaries: one corresponding to the first entrance into the emission region and another corresponding to the last relevant interaction before the photon escapes. These quantities are tracked per thread and later reduced across all threads to obtain global extrema. In addition, we record the absolute minimum photon time encountered across all rays, which defines the earliest emission time needed for the simulation.

After all pixels have been processed, the intensity map is scaled by the appropriate frequency factor, completing the radiative transfer step.

Finally, temporary arrays storing thread-local geodesics and timing information are discarded, and garbage collection is triggered to free memory before proceeding to the next stage of the calculation. This ensures that only the necessary global quantities are retained for subsequent slow-light processing.

In [9]:
# Calculate the camera position in native coordinates
Xcamera = MVec4(camera_position(ro, th, phi, params.a, params.Rout))

# Unitless frequency 
const freq_unitless = freq * HPL / (ME * CL * CL)

# Array that will hold the Intensity value for each pixel
midplane_crossings = zeros(Int, pixels_x, pixels_y)
nsteps = zeros(Int, pixels_x, pixels_y)

# Number of threads used in the calculation
const nthreads = Threads.nthreads() + 1

println("Allocating workspaces for $nthreads threads...")
# Number of maximum steps in the geodesic calculation
const maxnstep = 15000

# Allocating the scratchpad vector for each thread
#thread_trajs = [Vector{OfTrajM}(undef, maxnstep) for _ in 1:nthreads]
#for t in 1:nthreads
#    for k in 1:maxnstep
#        thread_trajs[t][k] = OfTrajM(
#            0.0, 
#            MVec4(undef), MVec4(undef), MVec4(undef), MVec4(undef)
#        )
#    end
#end

# Update thread allocation to use the new immutable OfTrajS
dummy_svec = @SVector zeros(4)
dummy_traj = OfTrajS(0.0, dummy_svec, dummy_svec, dummy_svec, dummy_svec)

thread_trajs = [Vector{OfTrajS}(undef, maxnstep) for _ in 1:nthreads]
for t in 1:nthreads
    for k in 1:maxnstep
        thread_trajs[t][k] = dummy_traj
    end
end

# This will hold the exact number of steps for each pixel.
const all_geodesics = Matrix{Vector{OfTrajS}}(undef, pixels_x, pixels_y)

# Allocate an array to hold the minimum time found by EACH thread
# Initialized to 0.0 because your photon times will be negative
thread_t0 = zeros(Float64, nthreads)

# tgeoi needs to find the maximum (closest to zero), so initialize very negative
thread_tgeoi = fill(-1e100, nthreads) 

# tgeof needs to find the minimum (most negative), so initialize at zero
thread_tgeof = zeros(Float64, nthreads)

p = Progress(
    pixels_x * pixels_y; 
    desc = "Raytracing Image...", 
    showspeed = true, 
    barlen = 30
)
ProgressMeter.ijulia_behavior(:clear)

println("Tracing Geodesics...")
Threads.@threads for i in 0:(pixels_x - 1)
    tid = Threads.threadid() 
    
    for j in 0:(pixels_y - 1)
        nstep, midplane_crossings[i+1,j+1] = get_pixel(
            thread_trajs[tid], i, j, Xcamera, 
            fovx, fovy, freq_unitless, 
            pixels_x, pixels_y, params.a, 
            Rh, params.Rout, Rstop, xoff, yoff
        ) 
        nsteps[i+1, j+1] = nstep
        # Save to permanent storage
        #all_geodesics[i + 1, j + 1] = deepcopy(thread_trajs[tid][1:nstep])

        all_geodesics[i + 1, j + 1] = thread_trajs[tid][1:nstep]

        final_step_time = thread_trajs[tid][nstep].X[1]
        if final_step_time < thread_t0[tid]
            thread_t0[tid] = final_step_time
        end

        # tgeoi and tgeof calculations following how ipole does it
        pixel_tgeoi = 1.0
        pixel_tgeof = 1.0
        for k in 1:nstep
            X = thread_trajs[tid][k].X
            K = thread_trajs[tid][k].Kcon
        
            log_r = X[2]          # must be log(r)
            t_coord = X[1]
            k_r = K[2]            # must match Kcon[1]
        
            if pixel_tgeoi > 0.0 && log_r < log(100.0)
                pixel_tgeoi = t_coord
            end
        
            if pixel_tgeof > 0.0 && log_r > log(100.0) && k_r < 0.0
                pixel_tgeof = t_coord
            end
        end
        final_step_time = thread_trajs[tid][nstep].X[1]
        
        if pixel_tgeoi < 0.0 && pixel_tgeoi > thread_tgeoi[tid]
            thread_tgeoi[tid] = pixel_tgeoi
        end
        if pixel_tgeof < 0.0 && pixel_tgeof < thread_tgeof[tid]
            thread_tgeof[tid] = pixel_tgeof
        elseif pixel_tgeof > 0.0 && final_step_time < thread_tgeof[tid]
            thread_tgeof[tid] = final_step_time
        end
        
        ProgressMeter.next!(
                p; 
                showvalues = [
                    (:thread_id, tid), 
                    (:pixel, "($i, $j)"), 
                    (:total_done, "$(i*pixels_y + j)/$(pixels_x * pixels_y)")
            ]
        )
    end
end

finish!(p);

t0 = minimum(thread_t0)
tgeof = minimum(thread_tgeof) # The most negative time in the active zone (Oldest file needed)
tgeoi = maximum(thread_tgeoi) # The least negative time in the active zone (Newest file needed)

println("Calculated t0 (absolute longest time): $t0")
println("Calculated tgeof (oldest active time): $tgeof")
println("Calculated tgeoi (newest active time): $tgeoi")

# Eliminate arrays from RAM
thread_to = nothing
thread_tgeoi = nothing
thread_tgeof = nothing
thread_trajs = nothing
GC.gc()

Raytracing Image... 100%|██████████████████████████████| Time: 0:00:05 ( 0.34 ms/it)
    thread_id: 5
        pixel: (79, 127)
   total_done: 10239/16384


Calculated t0 (absolute longest time): -1170.567649392201
Calculated tgeof (oldest active time): -1170.567649392201
Calculated tgeoi (newest active time): -909.3501994514284


## Slow-Light Image Construction and Radiative Transfer

The function `process_slowlight_images!` performs the final stage of the slow-light pipeline.

The procedure begins by determining the first image time relative to the simulation window, using the earliest geodesic interaction time and the slow-light configuration. Based on the photon propagation times and the chosen image cadence, a set of image frames is scheduled across the available temporal domain. Because multiple images may be active simultaneously, a buffer of image structures is allocated to hold partially computed frames.

Each image corresponds to a specific target emission time. For every pixel in the image, the precomputed geodesic trajectory is reused and integrated backwards in time through the evolving simulation. Along each trajectory, the code steps through spacetime points and reconstructs the radiative transfer by sampling the local plasma properties at the appropriate emission time. This requires dynamically adjusting the photon time coordinate to remain within the valid slow-light time window defined by the loaded simulation dumps.

Once all pixels of a frame are processed, the resulting intensity map is scaled by the appropriate frequency factor and written to disk as a time-stamped output file. Frames that are not yet complete remain in memory until sufficient simulation data has been loaded to finalize them.

After each batch of frames, the simulation window is advanced using `update_data!`, which loads the next GRMHD dump and shifts the slow-light interpolation window forward in time. This ensures that photon trajectories always have access to the correct plasma state as they traverse the evolving simulation.

The loop continues until all images have been fully constructed and no active frames remain, producing a complete time-dependent movie of the slow-light black hole emission.

In [ ]:
process_slowlight_images!(
    params_slowlight, 
    simulation_data, 
    all_geodesics, 
    nsteps, 
    params, 
    t0, 
    tgeof, 
    tgeoi, 
    pixels_x, 
    pixels_y, 
    freq
)

First Image will be produced at 2270.570329889598


Rendering frame slice 1... out of 119 100%|██████████████████████████████| Time: 0:00:01 (71.85 μs/it)
Rendering frame slice 2... out of 119 100%|██████████████████████████████| Time: 0:00:00 (41.18 μs/it)
Rendering frame slice 3... out of 119 100%|██████████████████████████████| Time: 0:00:00 (27.02 μs/it)
Rendering frame slice 4... out of 119 100%|██████████████████████████████| Time: 0:00:00 (18.38 μs/it)
Rendering frame slice 5... out of 119 100%|██████████████████████████████| Time: 0:00:00 ( 6.84 μs/it)


Loading data from '/home/pedro/kharma/iharm3d_out/tmp.00003.h5' into 'iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)


┌ Info: Loaded data
│   dump_path = "/home/pedro/kharma/iharm3d_out/tmp.00004.h5"
│   tA = 1200.0065090072605
└   tB = 1300.009885092488
Rendering frame slice 1... out of 119 100%|██████████████████████████████| Time: 0:00:03 ( 0.24 ms/it)
Rendering frame slice 2... out of 119 100%|██████████████████████████████| Time: 0:00:03 ( 0.24 ms/it)
Rendering frame slice 3... out of 119 100%|██████████████████████████████| Time: 0:00:03 ( 0.24 ms/it)
Rendering frame slice 4... out of 119 100%|██████████████████████████████| Time: 0:00:03 ( 0.23 ms/it)
Rendering frame slice 5... out of 119 100%|██████████████████████████████| Time: 0:00:03 ( 0.21 ms/it)
Rendering frame slice 6... out of 119 100%|██████████████████████████████| Time: 0:00:03 ( 0.19 ms/it)
Rendering frame slice 7... out of 119 100%|██████████████████████████████| Time: 0:00:02 ( 0.16 ms/it)
Rendering frame slice 8... out of 119 100%|██████████████████████████████| Time: 0:00:02 ( 0.13 ms/it)
Rendering frame slice 9... out of 119 1

Loading data from '/home/pedro/kharma/iharm3d_out/tmp.00004.h5' into 'iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)


┌ Info: Loaded data
│   dump_path = "/home/pedro/kharma/iharm3d_out/tmp.00005.h5"
│   tA = 1300.009885092488
└   tB = 1400.0083846507378
Rendering frame slice 1... out of 119 100%|██████████████████████████████| Time: 0:00:01 (92.70 μs/it)
Rendering frame slice 2... out of 119 100%|██████████████████████████████| Time: 0:00:01 ( 0.10 ms/it)
Rendering frame slice 3... out of 119 100%|██████████████████████████████| Time: 0:00:01 ( 0.12 ms/it)
Rendering frame slice 4... out of 119 100%|██████████████████████████████| Time: 0:00:02 ( 0.13 ms/it)
Rendering frame slice 5... out of 119 100%|██████████████████████████████| Time: 0:00:02 ( 0.15 ms/it)
Rendering frame slice 6... out of 119 100%|██████████████████████████████| Time: 0:00:02 ( 0.18 ms/it)
Rendering frame slice 7... out of 119 100%|██████████████████████████████| Time: 0:00:03 ( 0.21 ms/it)
Rendering frame slice 8... out of 119 100%|██████████████████████████████| Time: 0:00:03 ( 0.22 ms/it)
Rendering frame slice 9... out of 119 1